# 03 - Comandos DML em Tabelas Delta Lake

Demonstra operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de recursos como **HISTORY** e **TIME TRAVEL**.

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

## 2. Registrar Tabelas Delta como SQL Tables

In [ ]:
tabelas_delta = ['partidos', 'parlamentares', 'categorias_despesa', 'fornecedores', 'despesas']

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

## 3. Consultar Dados Atuais (SELECT)

In [ ]:
print('=== PARTIDOS ===')
spark.sql('SELECT * FROM partidos ORDER BY cd_partido').show()

print('=== PARLAMENTARES ===')
spark.sql('SELECT * FROM parlamentares ORDER BY cd_parlamentar').show()

print('=== DESPESAS (primeiras 10) ===')
spark.sql('SELECT * FROM despesas ORDER BY cd_despesa LIMIT 10').show()

In [ ]:
print(f'{"Tabela":<25} {"Registros":>10}')
print('-' * 37)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<25} {count:>10}')

---
## 4. INSERT - Inserir Novos Registros

In [ ]:
# INSERT - Novo partido
print('--- INSERT: Novo partido ---')
spark.sql('SELECT COUNT(*) as antes FROM partidos').show()

spark.sql("""
    INSERT INTO partidos VALUES
    (13, 'AVANTE', 'Avante'),
    (14, 'SOLIDARIEDADE', 'Solidariedade')
""")

spark.sql('SELECT * FROM partidos ORDER BY cd_partido').show()
print('2 novos partidos inseridos!')

In [ ]:
# INSERT - Novo parlamentar
print('--- INSERT: Novo parlamentar ---')

spark.sql("""
    INSERT INTO parlamentares VALUES
    (26, 'Zenobio Machado Araujo', 'DF', 13, 57),
    (27, 'Wanda Pereira dos Santos', 'RR', 14, 57)
""")

spark.sql('SELECT * FROM parlamentares WHERE cd_parlamentar >= 26').show()
print('2 novos parlamentares inseridos!')

In [ ]:
# INSERT - Nova despesa
print('--- INSERT: Nova despesa ---')

spark.sql("""
    INSERT INTO despesas VALUES
    (31, 26, 1, 2, 6, 2024, '2024-06-05', 75.00, 0.00, 75.00, 'NF-018', 0),
    (32, 26, 4, 8, 6, 2024, '2024-06-10', 199.90, 0.00, 199.90, 'FAT-006', 0),
    (33, 27, 5, 3, 6, 2024, '2024-06-15', 980.00, 0.00, 980.00, 'ETICKET-006', 2)
""")

spark.sql('SELECT * FROM despesas WHERE cd_despesa >= 31').show()
print('3 novas despesas inseridas!')

---
## 5. UPDATE - Atualizar Registros

In [ ]:
# UPDATE - Corrigir partido de parlamentar via SQL
print('--- UPDATE: Parlamentar muda de partido (migracao partidaria) ---')
print('ANTES:')
spark.sql('SELECT cd_parlamentar, nm_parlamentar, cd_partido FROM parlamentares WHERE cd_parlamentar = 5').show()

spark.sql("""
    UPDATE parlamentares
    SET cd_partido = 3
    WHERE cd_parlamentar = 5
""")

print('DEPOIS:')
spark.sql('SELECT cd_parlamentar, nm_parlamentar, cd_partido FROM parlamentares WHERE cd_parlamentar = 5').show()

In [ ]:
# UPDATE - Corrigir valor de despesa (glosa aplicada) via SQL
print('--- UPDATE: Aplicar glosa em despesa ---')
print('ANTES:')
spark.sql('SELECT cd_despesa, vl_documento, vl_glosa, vl_liquido FROM despesas WHERE cd_despesa = 10').show()

spark.sql("""
    UPDATE despesas
    SET vl_glosa = 500.00,
        vl_liquido = 2000.00
    WHERE cd_despesa = 10
""")

print('DEPOIS:')
spark.sql('SELECT cd_despesa, vl_documento, vl_glosa, vl_liquido FROM despesas WHERE cd_despesa = 10').show()

In [ ]:
# UPDATE via DeltaTable API - Atualizar nome de fornecedor
print('--- UPDATE via DeltaTable API: Atualizar razao social do fornecedor ---')
from pyspark.sql.functions import lit

dt_fornecedores = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/fornecedores')

print('ANTES:')
spark.sql('SELECT * FROM fornecedores WHERE cd_fornecedor = 3').show()

dt_fornecedores.update(
    condition="cd_fornecedor = 3",
    set={"nm_fornecedor": lit("LATAM Airlines Brasil S.A. (Incorporada pela GOL)")
    }
)

print('DEPOIS:')
spark.sql('SELECT * FROM fornecedores WHERE cd_fornecedor = 3').show()

---
## 6. DELETE - Remover Registros

In [ ]:
# DELETE - Remover despesas com glosa total (vl_liquido = 0) via SQL
# Simula o cancelamento de reembolsos integralmente glosados
print('--- DELETE: Remover despesas totalmente glosadas ---')

# Primeiro inserimos uma despesa totalmente glosada para demonstrar
spark.sql("""
    INSERT INTO despesas VALUES
    (34, 1, 2, 1, 6, 2024, '2024-06-20', 400.00, 400.00, 0.00, 'NF-019', 0)
""")

print('Despesas com vl_liquido = 0 (totalmente glosadas):')
spark.sql('SELECT * FROM despesas WHERE vl_liquido = 0').show()

spark.sql('DELETE FROM despesas WHERE vl_liquido = 0')

print('Após DELETE — despesas com vl_liquido = 0:')
spark.sql('SELECT * FROM despesas WHERE vl_liquido = 0').show()

In [ ]:
# DELETE - Remover parlamentares de teste via SQL
print('--- DELETE: Remover parlamentar inserido ---')
spark.sql('SELECT * FROM parlamentares WHERE cd_parlamentar = 27').show()

spark.sql('DELETE FROM parlamentares WHERE cd_parlamentar = 27')

print('Após DELETE:')
spark.sql('SELECT * FROM parlamentares WHERE cd_parlamentar >= 26').show()

In [ ]:
# DELETE via DeltaTable API - Remover partido de teste
print('--- DELETE via DeltaTable API: Remover partido ---')
dt_partidos = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/partidos')

print('Antes:')
spark.sql('SELECT * FROM partidos WHERE cd_partido = 14').show()

dt_partidos.delete('cd_partido = 14')

print('Depois:')
spark.sql('SELECT * FROM partidos WHERE cd_partido = 14').show()

---
## 7. HISTORY - Histórico de Versões Delta

O Delta Lake mantém um log de transações que permite visualizar o histórico completo de alterações — fundamental para auditoria de dados públicos.

In [ ]:
# Histórico da tabela despesas (mais relevante para dados parlamentares)
print('=== HISTÓRICO DA TABELA DESPESAS ===')
dt_despesas = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/despesas')
dt_despesas.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

In [ ]:
# Histórico via Spark SQL
print('=== HISTÓRICO DA TABELA PARLAMENTARES ===')
spark.sql('DESCRIBE HISTORY parlamentares').select('version', 'timestamp', 'operation').show(truncate=False)

---
## 8. TIME TRAVEL - Viagem no Tempo

O Delta Lake permite ler versões anteriores — útil para auditar alterações em dados de gastos públicos.

In [ ]:
# Versão atual da tabela despesas
print('=== DESPESAS - VERSÃO ATUAL ===')
spark.sql('SELECT COUNT(*) as total_despesas, SUM(vl_liquido) as total_liquido FROM despesas').show()

In [ ]:
# Time Travel - Ler versão 0 (estado original da carga)
print('=== DESPESAS - VERSÃO 0 (original, antes dos DMLs) ===')
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/despesas')
df_v0.selectExpr('COUNT(*) as total', 'SUM(vl_liquido) as total_liquido').show()
df_v0.show()

In [ ]:
# Comparação entre versão original e atual
print('=== COMPARAÇÃO: Versão 0 vs Atual ===')
df_original = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/despesas')
df_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/despesas')

print(f'Versão 0: {df_original.count()} despesas')
print(f'Atual:    {df_atual.count()} despesas')

print('\nDespesas ADICIONADAS (presentes no atual, ausentes na v0):')
df_atual.subtract(df_original).show()

print('\nDespesas REMOVIDAS ou ALTERADAS (presentes na v0, ausentes no atual):')
df_original.subtract(df_atual).show()

---
## 9. Resumo Final

In [ ]:
print('=' * 65)
print('RESUMO DAS OPERAÇÕES DML REALIZADAS')
print('=' * 65)
print()
print('INSERT:')
print('  - 2 novos partidos (AVANTE, SOLIDARIEDADE)')
print('  - 2 novos parlamentares (DF e RR)')
print('  - 3 novas despesas do parlamentar 26')
print()
print('UPDATE:')
print('  - parlamentar 5: migração partidária MDB → UNIÃO (via SQL)')
print('  - despesa 10: glosa aplicada, vl_liquido reduzido (via SQL)')
print('  - fornecedor 3: razão social atualizada (via DeltaTable API)')
print()
print('DELETE:')
print('  - despesa 34: cancelada por glosa total (vl_liquido = 0)')
print('  - parlamentar 27: removido (via SQL)')
print('  - partido 14: removido (via DeltaTable API)')
print()
print('HISTORY e TIME TRAVEL:')
print('  - Histórico completo de transações exibido')
print('  - Comparação versão 0 (original) vs atual')
print('  - Rastreabilidade total de alterações nos dados públicos')
print('=' * 65)

In [ ]:
spark.stop()
print('SparkSession finalizada.')